# Intro

This notebook spins an instance in Kaggle, runs ollama and sets an ngrok tunnel up for accessing the gemma4-31b (q4) model. You can then use the model for all sorts of stuff: chat, coding, openclaw etc.

## Manual Setup
* Settings/Accelerator/GPU T4x2
* Add-ons/Secrets/NGROK_AUTHTOKEN   (https://dashboard.ngrok.com/get-started/your-authtoken)
  
## endpoint to access the gemma4-31b model
* Visit https://dashboard.ngrok.com/endpoints

# check GPU

In [1]:
from rich import print
# !nvidia-smi
# !test -e /dev/nvidia0 && echo "GPU present" || echo "No GPU"

no_gpu = !test -e /dev/nvidia0
if no_gpu:
    print("[bold red]No GPU. Turn on GPU and try again, e.g. Settings/Accelerator/GPU T4x2")
    raise SystemExit(1)
else:
    print("[bold green]GPU found, good to go")


GPU found, good to go


# Setup pip

In [2]:
try:
    from set_env import set_env
except ModuleNotFoundError:
    !uv pip install --system -q pyngrok set-env-colab-kaggle-dotenv
    from set_env import set_env

# set to a timezone you prefer
%env TZ=Asia/Shanghai 
from rich import print


env: TZ=Asia/Shanghai


# setup env var NGROK_AUTHTOKEN 

In [3]:
import os
from set_env import set_env

# set a Secret NGROK_AUTHTOKEN in Add-ons/Screts or you can paste NGROK_AUTHTOKEN when asked below
set_env("NGROK_AUTHTOKEN")

if os.getenv("NGROK_AUTHTOKEN"):
    print("[bold green]NGROK_AUTHTOKEN set, good to go")
else:
    NGROK_AUTHTOKEN = input("paste NGROK_AUTHTOKEN here and hit Enter: ")
    os.environ['NGROK_AUTHTOKEN'] = NGROK_AUTHTOKEN
    
    if not os.getenv("NGROK_AUTHTOKEN"):
        print("[bold red]set NGROK_AUTHTOKEN and try again")
        raise SystemExit(1)
    else:
        print("[bold green]NGROK_AUTHTOKEN set, good to go")

NGROK_AUTHTOKEN set, good to go

# Setup tunnel

In [4]:

from IPython.display import clear_output
# get_ipython().system_raw('OLLAMA_ORIGINS="*" ollama serve > ollama.log 2>&1 &')

import os
# import subprocess
from pyngrok import ngrok

port = "11434"
tunnel = ngrok.connect(port)

print(f"ngroktunnel: {tunnel}")

# clickable url, it's ok when 'Your Service' is not conncted since ollama is not running yet
display(f"ngroktunnel: {tunnel}")

ngroktunnel: NgrokTunnel: "https://punk-glitch-backlight.ngrok-free.dev" -> "http://localhost:11434"

'ngroktunnel: NgrokTunnel: "https://punk-glitch-backlight.ngrok-free.dev" -> "http://localhost:11434"'

In [5]:
try:
    from loguru import logger
except ModuleNotFoundError:
    !uv pip install loguru
    from loguru import logger
logger.info(f"ngroktunnel: {tunnel=}")
logger.info(f"{'=' * 20}")


2026-04-15 12:47:15.645 | INFO     | __main__:<cell line: 0>:6 - ngroktunnel: tunnel=<NgrokTunnel: "https://punk-glitch-backlight.ngrok-free.dev" -> "http://localhost:11434">
2026-04-15 12:47:15.646 | INFO     | __main__:<cell line: 0>:7 - ====================


# start ollama

In [6]:
output = !which ollama
if 'ollama' not in str(output):
    ! apt-get install zstd
    !curl -fsSL https://ollama.com/install.sh | sh
output = !which ollama
f'{output=}'

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (582 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
###############################################                           66.3%^C########                                          44.5%#################################

"output=['/usr/local/bin/ollama']"

In [7]:
%%time
import os
from time import sleep, time

output = !which ollama
if not output:
    !curl -fsSL https://ollama.ai/install.sh | sh
    clear_output()

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'

output = !ps aux|grep ollama |grep -vE "grep|defunct"

then = time()
wait_dur = 120
print('[green]waiting for ollama to start...')

# subprocess.Popen(["ollama", "serve"]
get_ipython().system_raw("ollama serve > ollama.log 2>&1 &")
o = !curl -s 127.0.0.1:11434

while time() - then < wait_dur:
    if "running" in str(o):
        print(f'[bold green]{o}')
        break
    sleep(2)
    # print('.', end="")
    o = !curl -s 127.0.0.1:11434
else:
    print(f"[bold red]ollama is not ready after {wait_dur} s, may need extra attention")    

waiting for ollama to start...

KeyboardInterrupt: 

In [8]:
!tail ollama.log

time=2026-04-15T21:13:42.499+08:00 level=INFO source=server.go:444 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 39161"
time=2026-04-15T21:13:42.499+08:00 level=INFO source=server.go:444 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 34489"
time=2026-04-15T21:13:42.499+08:00 level=INFO source=server.go:444 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 33039"
time=2026-04-15T21:13:42.499+08:00 level=INFO source=server.go:444 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 44261"
time=2026-04-15T21:13:42.539+08:00 level=INFO source=runner.go:464 msg="failure during GPU discovery" OLLAMA_LIBRARY_PATH="[/usr/local/lib/ollama /usr/local/lib/ollama/cuda_v13]" extra_envs="map[CUDA_VISIBLE_DEVICES:GPU-5b2ab759-4c08-04e2-92aa-3a169e32f26c GGML_CUDA_INIT:1]" error="failed to finish discovery before timeout"
time=2026-04-15T21:13:42.539+08:00 level=INFO source=runner.go

In [9]:
o = !curl -s 127.0.0.1:11434
print(['running' in str(o), o])
assert 'running' in str(o), 'ollama is not running, fix it and try again'

[False, []]

AssertionError: ollama is not running, fix it and try again

In [ ]:
# !ollama list
!ps aux|grep "ollama serve" | grep -v grep

# Donwload Model

In [ ]:

# print('[bold green] need to download 15G model in the background')
# !ollama pull devstral:latest 
# clear_output()
# !ollama pull devstral:latest > pull.log
model_name = "devstral:latest"
# devstral-small-2 devstral-small-2:24b devstral-small-2:24b-cloud
model_name = "devstral-small-2:24b"
model_name = "gemma4:e4b"

print(f'[bold green] need to download {model_name=} in the background')

get_ipython().system_raw(f"ollama pull {model_name} > pull.log 2>&1 &")


In [ ]:
o = !tail pull.log
while '100%' not in str(o):
    o = !tail pull.log
    sleep(1)
o

In [ ]:
o = !ollama list
if "to start it" in str(o):
    # !ollama serve
    get_ipython().system_raw("ollama serve > ollama.log 2>&1 &")
!ollama list

In [ ]:
%%time

from time import sleep, time
o = !ollama list
then = time()
wait_dur = 300

# make sure ollama is running
_ = !curl 127.0.0.1:11434
assert o, "ollama is not running, fix it and try again"

while time() - then < wait_dur:
    if model_name in str(o):
        break
    sleep(1)
    o = !ollama list

print(f'{o}')
!echo {o}

In [ ]:
model_name = "nomic-embed-text"

print(f'[bold green] need to download {model_name=} in the background')

get_ipython().system_raw(f"ollama pull {model_name} > pull.log 2>&1 &")

In [ ]:
o = !tail pull.log
while '100%' not in str(o):
    o = !tail pull.log
    sleep(1)
o

In [ ]:
!ollama list

# base_url/api_key info

In [ ]:

!curl 127.0.0.1:11434
print("")
print(f'{tunnel}, click the NgrokTunnel url to make sure the external API is working (should display "Ollama is running")')

try:
    public_url = tunnel.public_url
except Exception as exc:
    public_url = ''
    raise SystemExit('ngrok tunnel not properly setup, fix that and try again')

print(f"base_url: {public_url}/v1, api_key: any (for example ollam)")

curl_cmd = f'''curl -sSLk {public_url}/v1/models -H "Authorization: Bearer any"'''
print(f"Run {curl_cmd} (output: {model_name})")


In [ ]:
!echo echo {curl_cmd}

logger.info(f"logger.info {curl_cmd}")

import sys
print(f"print file=sys.stderr {curl_cmd}", file=sys.stderr)

In [ ]:
# display(curl_cmd)
import sys

print(f'[green]{public_url} {curl_cmd}')

# show in 
print(f'[green]{public_url} {curl_cmd}', file=sys.stderr)

In [ ]:
# clickable
f'{tunnel.public_url}/v1/models'

# check ollama status

In [ ]:
# !ollama list
!curl 127.0.0.1:11434

# should display 'Ollama is running' or a model, if not go back and restart ollama serve 

# subprocess.Popen("ollama serve", shell=True)

# check ngrok tunnel

In [ ]:

!ps aux|grep -i ngrok | grep -vE "grep|defunct" | sed 's/--authtoken.*/--authtoken [REDACTED]/'

!printf {public_url}

# should display one entry 

# check ollama.log


In [ ]:
!tail -f ollama.log